##  Statistiques descriptives
   - Nombre d'avis par classe
   - Longueur moyenne, min, max
   - Distribution des longueurs
   - Vocabulaire total unique

In [ ]:
# Calcul des statistiques de base
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

# Conversion en sentiment binaire (0 = négatif, 1 = positif)
df['sentiment'] = df['label'].map({1: 0, 2: 1})
df['sentiment_str'] = df['sentiment'].map({0: 'négatif', 1: 'positif'}) 

print(f"\n Nombre total d'avis : {len(df):,}")
print(f" Nombre de colonnes : {len(df.columns)}")

print("\n--- Distribution des Classes ---")
class_dist = df['sentiment_str'].value_counts()
class_pct = df['sentiment_str'].value_counts(normalize=True) * 100
for sentiment in class_dist.index:
    print(f"   {sentiment.capitalize()}: {class_dist[sentiment]:,} ({class_pct[sentiment]:.1f}%)")

print("\n--- Statistiques de Longueur des Textes ---")
print(f"   Longueur moyenne (caractères) : {df['text_length'].mean():.1f}")
print(f"   Longueur médiane (caractères) : {df['text_length'].median():.1f}")
print(f"   Longueur min : {df['text_length'].min()}")
print(f"   Longueur max : {df['text_length'].max():,}")

print("\n--- Statistiques du Nombre de Mots ---")
print(f"   Nombre moyen de mots : {df['word_count'].mean():.1f}")
print(f"   Nombre médian de mots : {df['word_count'].median():.1f}")
print(f"   Minimum : {df['word_count'].min()}")
print(f"   Maximum : {df['word_count'].max()}")    

In [ ]:
# Visualisation de la distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Distribution des sentiments
colors = ['#FF6B6B', '#4ECDC4']
ax1 = axes[0]
bars = ax1.bar(class_dist.index, class_dist.values, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_title('Distribution des Sentiments', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sentiment')
ax1.set_ylabel('Nombre d\'avis')
for bar, val in zip(bars, class_dist.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
             f'{val:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Graphique 2 : Ratio des classes (Pie chart)
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(class_dist.values, labels=class_dist.index, 
                                   autopct='%1.1f%%', colors=colors,
                                   explode=(0.02, 0.02), shadow=True,
                                   textprops={'fontsize': 12})
ax2.set_title('Proportion des Sentiments', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/figures/01_distribution_sentiments.png', dpi=150, bbox_inches='tight')
plt.show()

# Vérification du déséquilibre
ratio = max(class_dist.values) / min(class_dist.values)
if ratio > 1.5:
    print(f"  Attention : déséquilibre de classes détecté (ratio {ratio:.2f}:1)")
    print("   Considérez des techniques de rééquilibrage (oversampling, undersampling, class_weight)")
else:
    print(f" classes relativement équilibrées (ratio {ratio:.2f}:1)")

In [ ]:
# Distribution des longueurs par sentiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme des longueurs
ax1 = axes[0]
for sentiment, color, label in zip([0, 1], colors, ['Négatif', 'Positif']):
    subset = df[df['sentiment'] == sentiment]['text_length']
    ax1.hist(subset, bins=50, alpha=0.6, color=color, label=label, edgecolor='black')
ax1.set_title('Distribution de la Longueur des Textes par Sentiment', fontsize=12, fontweight='bold')
ax1.set_xlabel('Longueur (caractères)')
ax1.set_ylabel('Fréquence')
ax1.legend()
ax1.set_xlim(0, df['text_length'].quantile(0.99))  # Exclure les outliers extrêmes

# Boxplot comparatif
ax2 = axes[1]
df.boxplot(column='word_count', by='sentiment_str', ax=ax2)
ax2.set_title('Nombre de Mots par Sentiment', fontsize=12, fontweight='bold')
ax2.set_xlabel('Sentiment')
ax2.set_ylabel('Nombre de mots')
plt.suptitle('')  # Supprimer le titre automatique

plt.tight_layout()
plt.savefig('figures/02_distribution_longueurs.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques par sentiment
print("\n Statistiques par sentiment :")
print(df.groupby('sentiment_str')[['text_length', 'word_count']].describe().round(1))